
# Large Language Models (LLMs)

## Evolution of LLMs

The field of NLP has evolved rapidly—and **scale** has been the key driver.

| Model       | Year | Parameters  | Training Data | Notable Achievement                  |
|----------- |---- |----------- |------------- |------------------------------------ |
| **BERT**    | 2018 | 340M        | 16GB          | Revolutionized "understanding" tasks |
| **GPT-2**   | 2019 | 1.5B        | 40GB          | "Too dangerous to release"           |
| **GPT-3**   | 2020 | 175B        | 570GB         | Emergent few-shot abilities          |
| **Llama-2** | 2023 | 70B         | 2TB           | Open-weight ecosystem growth         |
| **GPT-4**   | 2023 | Undisclosed | Unknown       | Multimodal (text + images)           |

Key architectures:

1.  **BERT (Encoder-only)**: Good at "understanding" (Classification, Q&A). Reads text and produces vectors.
2.  **GPT (Decoder-only)**: Good at "generation" (Writing text). Predicts the next token.
3.  **T5 (Encoder-Decoder)**: Treats everything as "text-to-text" (Translation, Summarization).

**The Scaling Hypothesis**: Performance improves predictably with more parameters, more data, and more compute. This drove the race from millions to trillions of parameters.

## The Lifecycle of an LLM

1.  **Pre-training**: The expensive part. The model reads the internet (TB of text) and learns to predict the next token. It learns grammar, facts, and reasoning.
2.  **Fine-tuning**: The specialization part.
    -   **SFT (Supervised Fine-Tuning)**: Train on Q&A pairs to make it a chatbot instead of a document completer.
    -   **RLHF (Reinforcement Learning from Human Feedback)**: Aligning the model to be helpful and harmless.
3.  **Inference**: Using the model. We feed a prompt, and it generates the response.

## Tokenization & Context Windows

LLMs do not see words; they see **Tokens**.

-   A token can be a word ("apple"), a part of a word ("ing"), or a character ("a").
-   **Rough rule**: 1000 tokens $\approx$ 750 words.
-   **Context Window**: The limit on how much text the model can remember at once (varies by model/version).

## Practical Demonstration: Tokenization

We will use `transformers` to see how text is broken down.

In [ ]:
from transformers import AutoTokenizer

# Load a standard tokenizer (GPT-2 style)
tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "LLMs are fascinating!"

# 1. Encode (Text -> Integers)
tokens = tokenizer.encode(text)
print(f"Token IDs: {tokens}")

# 2. Decode (Integers -> Text)
decoded = [tokenizer.decode([t]) for t in tokens]
print(f"Token Parts: {decoded}")

### Why Subwords? Common vs Rare Words

Tokenizers use **Byte Pair Encoding (BPE)** to handle any word, even ones never seen in training.

In [ ]:
# Compare token counts for different words
test_words = ["hello", "AI", "Anthropic", "TensorFlow", "🤖"]

print("Word -> Token Count -> Subwords")
print("-" * 40)
for word in test_words:
    tokens = tokenizer.encode(word)
    decoded = [tokenizer.decode([t]) for t in tokens]
    print(f"'{word}' -> {len(tokens)} tokens: {decoded}")

**Observation**: Common words like "hello" are single tokens. Rare words like "Anthropic" get split into subwords. This is why LLMs can handle **any** text, even made-up words.

## Practical Demonstration: Using a Pre-trained LLM

We will use a small GPT-2 model to generate text. (We use GPT-2 because it fits in memory easily; modern models like Llama-3 require GPUs).

In [ ]:
from transformers import AutoModelForCausalLM
import torch

# Load Model
model = AutoModelForCausalLM.from_pretrained("gpt2")

# Prepare Input
prompt = "The future of AI is"
inputs = tokenizer(prompt, return_tensors="pt")

# Generate
# max_new_tokens=20: Generate 20 tokens
# do_sample=True: Add randomness (creativity)
# temperature=0.7: Controls randomness (Lower = More focused)
output = model.generate(
    **inputs, 
    max_new_tokens=20, 
    do_sample=True, 
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id
)

result = tokenizer.decode(output[0])
print(f"Prompt: {prompt}")
print(f"Generated: {result}")

### Seeing the Probability Distribution

Let's peek inside and see what the model **actually** considers as next-token candidates.

In [ ]:
import torch.nn.functional as F

# Get model's raw predictions (logits) for our prompt
with torch.no_grad():
    outputs = model(inputs.input_ids)
    # Logits for the LAST position (predicting next token)
    logits = outputs.logits[0, -1, :]

# Convert to probabilities
probs = F.softmax(logits, dim=0)

# Get top 5 candidates
top_probs, top_indices = torch.topk(probs, 5)

print(f"Prompt: '{prompt}'")
print(f"\nTop 5 next-token predictions:")
for prob, idx in zip(top_probs, top_indices):
    token = tokenizer.decode([idx])
    print(f"  '{token}': {prob.item()*100:.1f}%")

**Key Insight**: The model doesn't "choose" one word—it produces a probability distribution over **all** 50,000+ tokens. Generation is just sampling from this distribution repeatedly.

## Prompt Engineering Basics

Since LLMs are just "next token predictors," they are sensitive to how you ask.

1.  **Zero-Shot**: Just asking. **"Translate to Spanish: Hello"**
2.  **Few-Shot**: Giving examples. **"English: Red, Spanish: Rojo"** **"English: Blue, Spanish: Azul"** **"English: Green, Spanish: &#x2026;"**
3.  **Chain of Thought**: Asking the model to "think step by step." This often helps on multi-step tasks, but is model/task dependent.

## Exercises

### Temperature Playground

Temperature controls the "risk" the model takes.

-   Generate text with `temperature=0.1` (Deterministic, boring).
-   Generate text with `temperature=1.5` (Chaotic, creative/nonsense).

### Sampling Strategies: Top-k and Top-p

Temperature alone isn't enough. Modern LLMs use additional sampling strategies:

| Parameter       | What it does                                           | Typical Value |
|--------------- |------------------------------------------------------ |------------- |
| **Temperature** | Sharpens or flattens the probability distribution      | 0.7           |
| **Top-k**       | Only consider the k most likely tokens                 | 50            |
| **Top-p**       | Only consider tokens until cumulative probability >= p | 0.9           |

-   **Top-k=50**: "Only consider the 50 most likely next words."
-   **Top-p=0.9** (Nucleus Sampling): "Consider words until their probabilities sum to 90%."

### Few-Shot Prompting

Try to make GPT-2 solve a pattern matching task.

-   Without examples, it might just continue the sentence randomly.
-   With examples, it should follow the pattern.

### Chain of Thought (CoT)

Adding "Let's think step by step" can improve performance on some multi-step problems, depending on the model.

-   **Without CoT**: "What is 17 × 24?" → Model guesses (often wrong).
-   **With CoT**: "What is 17 × 24? Let's think step by step." → Model shows working.

**Key Point**: CoT can provide useful intermediate "scratch space," but gains are not guaranteed and depend on model capability and prompt quality.

## Summary

1.  **LLMs**: Predict the next token based on vast training data.
2.  **Architecture**: BERT (Encoder) for understanding, GPT (Decoder) for writing.
3.  **Tokens**: The atoms of language models. Not always full words.
4.  **Prompting**: The skill of guiding the predictor to the output you want by providing context and examples.